In [13]:
import numpy as np
import pandas as pd

In [5]:
# loading data-set
raw_data=pd.read_csv("ncr_ride_bookings.csv")


# ========================================================================== 
# =========================================================================|| 
# extracted main data ( only which should be used )                       #||
booking_date = pd.to_datetime(raw_data['Date']).to_numpy()                #||
booking_time = pd.to_timedelta(raw_data['Time']).to_numpy()               #||
                                                                          #||
booking_status = raw_data['Booking Status'].to_numpy(dtype=str)           #||
                                                                          #||
booking_value = raw_data['Booking Value'].to_numpy(dtype=np.float64)      #||
ride_distance = raw_data['Ride Distance'].to_numpy(dtype=np.float64)      #||
                                                                          #||
driver_rating = raw_data['Driver Ratings'].to_numpy(dtype=np.float64)     #||
customer_rating = raw_data['Customer Rating'].to_numpy(dtype=np.float64)  #||
# =========================================================================||
# ==========================================================================



# parameter to work:-
# booking_date  booking_time booking_status booking_value ride_distance driver_rating customer_rating

## 🟢 Phase 1: Data Imputation & State Extraction
Task 1.1: Impute missing values (NaN) in both the driver_rating and customer_rating arrays, setting them to 0.0.

Task 1.2: Calculate the exact total count of rides where the booking_status is strictly "Completed".

In [21]:
# reaplaced nAn values to integer 
booking_value = np.nan_to_num(booking_value,nan=0.0)
ride_distance = np.nan_to_num(ride_distance,nan=0.0)
driver_rating = np.nan_to_num(driver_rating,nan=0.0)
customer_rating = np.nan_to_num(customer_rating,nan=0.0)


true_ride = np.where(booking_status=="Completed",1,0)
print(np.sum(true_ride))


93000


## 🟡 Phase 2: Metric Derivation & Dimensional Matrix Construction
Task 2.1: Generate a new 1D array named revenue_per_km by dividing booking_value by ride_distance. You must mathematically handle zero-distance trips to output 0.0 instead of yielding a division-by-zero (Inf/NaN) error.

Task 2.2: Construct a new 2D consolidated array named analytics_matrix by stacking booking_value, ride_distance, and revenue_per_km horizontally as distinct columns.

In [ ]:
# taske : 2.1
revenue_per_km = np.where(ride_distance > 0, booking_value / ride_distance, 0.0)
print("Revenue Array Shape:", revenue_per_km.shape)


# Task 2.2 
analytics_matrix = np.column_stack((booking_value, ride_distance, revenue_per_km))
print("Matrix Shape:", analytics_matrix.shape) # Should be (150000, 3)


Revenue Array Shape: (150000,)
Matrix Shape: (150000, 3)


C:\Users\mycom\AppData\Local\Temp\ipykernel_9992\3454095960.py:2: RuntimeWarning: invalid value encountered in divide
  revenue_per_km = np.where(ride_distance > 0, booking_value / ride_distance, 0.0)


## 🟠 Phase 3: Multi-dimensional Filtering & Anomaly Detection
Task 3.1: Determine the total count of valid high-tier rides. A ride is defined as high-tier if it meets all three conditions: booking_value > 500, ride_distance > 15.0, AND booking_status is "Completed".

Task 3.2 (Anomaly Audit): Identify potential rating manipulation or severe dissatisfaction. Find the total number of rides where the absolute difference between driver_rating and customer_rating is 2.0 or greater.

In [36]:
# task 3.1 solution: (here we have used bit-wise & operator)
high_tier_ride = (booking_status =="Completed") & (booking_value >500) & (ride_distance >15)
count_high_tier_ride = np.sum(high_tier_ride)
print(f"Total high-tier-rides: {count_high_tier_ride}")

ratings = driver_rating - customer_rating
total_ratings = (ratings >= 2) | (ratings <= -2)
print(f"Potential ratings: {np.sum(total_ratings)}")

Total high-tier-rides: 26134
Potential ratings: 62


### 🔴 Phase 4: Statistical Aggregation & Outlier Isolation
Task 4.1: Calculate the net_valid_revenue, which is the total sum of booking_value isolated strictly to rides with a "Completed" status.

Task 4.2: Identify extreme distance outliers. Isolate and count the number of rides where the ride_distance is strictly greater than the 99th percentile of the dataset's overall distance array.

In [ ]:
# Task 4.1
print(f"Total valid revenue: {np.sum(booking_value[booking_status=="Completed"])}")
# i didn't know about the percentile function and how to use it so both are here
t99_percentile = np.percentile(ride_distance,99)
print(f"total rides > 99th percentile : {ride_distance[ride_distance > t99_percentile].size}")
print(f"total rides > 99th percentile : {np.sum(ride_distance>t99_percentile)}")


Total valid revenue: 47260574.0
total rides > 99th percentile : 1496
total rides > 99th percentile : 1496


### 🔥 Phase 5: Performance Ranking & Index Mapping
Task 5.1: Extract the original dataset index positions of the Top 10 highest-grossing rides based on booking_value.

Task 5.2: Using the exact index mappings extracted in Task 5.1, dynamically slice the arrays to print the booking_status and ride_distance exclusively for those Top 10 grossing rides.


In [72]:
sorted_index = np.argsort(booking_value)
t10_index = sorted_index[ : : -1][:10]


print(f"top 10 indices: \n{t10_index}")
print(f"Booking value: \n{booking_value[t10_index]}")
print(f"Booking status: \n{booking_status[t10_index]}")
print(f"Ride distance: \n{ride_distance[t10_index]}")

top 10 indices: 
[  7450  83506   2731  90418   7116  43469  21011  32935 117985  90667]
Booking value: 
[4277. 4228. 4220. 4202. 4133. 4109. 4093. 4088. 4060. 4044.]
Booking status: 
['Completed' 'Completed' 'Completed' 'Completed' 'Completed' 'Completed'
 'Completed' 'Completed' 'Completed' 'Completed']
Ride distance: 
[ 8.66 11.73 10.11  4.62 25.66 36.81 20.85 46.36 42.72  4.97]


In [65]:
# Task 5.1: Get indexes of top 10 highest booking values
sorted_indices = np.argsort(booking_value)
top_10_indices = sorted_indices[::-1][:10] # Reverse karke starting ke 10 uthaye!

print("Top 10 Ride Indexes:", top_10_indices)
print("-" * 40)

# Task 5.2: Print exact details using fancy indexing
print(f"Top 10 Booking Values: \n{booking_value[top_10_indices]}")
print(f"Top 10 Booking Status: \n{booking_status[top_10_indices]}")
print(f"Top 10 Ride Distances: \n{ride_distance[top_10_indices]}")

Top 10 Ride Indexes: [  7450  83506   2731  90418   7116  43469  21011  32935 117985  90667]
----------------------------------------
Top 10 Booking Values: 
[4277. 4228. 4220. 4202. 4133. 4109. 4093. 4088. 4060. 4044.]
Top 10 Booking Status: 
['Completed' 'Completed' 'Completed' 'Completed' 'Completed' 'Completed'
 'Completed' 'Completed' 'Completed' 'Completed']
Top 10 Ride Distances: 
[ 8.66 11.73 10.11  4.62 25.66 36.81 20.85 46.36 42.72  4.97]
